# Chess playing engine

## Imports and global variables

In [ ]:
import gc
import io
import json
import subprocess
import warnings
from pathlib import Path

import numpy as np
import zstandard as zstd
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.utils.data
import torch.optim as optim
import torch.nn.functional as F

import torch_geometric
from torch_geometric.data import HeteroData, Batch

import chess
import chess.pgn

warnings.filterwarnings("ignore")

In [ ]:
print(f"  PyTorch: {torch.__version__}")
print(f"  PyG: {torch_geometric.__version__}")
print(f"  CUDA Available: {torch.cuda.is_available()}")
print(f"  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"  python-chess: {chess.__version__}")

In [ ]:
SEED = 42
PGN_PATH = ''
JSONL_PATH = ''
BATCH_SIZE = 32
NUM_WORKERS = 4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LEARNING_RATE = 3e-4
NUM_EPOCHS    = 30
VALUE_WEIGHT  = 1.0
POLICY_WEIGHT = 1.0
BATCH_SIZE    = 128 
CHECKPOINT_DIR = Path('')
CHECKPOINT_DIR.mkdir(exist_ok=True)
HIDDEN_DIM = 384
NUM_LAYERS = 6
NUM_HEADS  = 8     
NUM_EXPERTS = 4    
EXPERT_DIM  = 384

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

## Processing and preparing data
For that step it is necessary to download chess data from [lichess](https://database.lichess.org/) and install stockfish engine ('apt install stockfish')

In [ ]:
class PGNProcessor:
    def __init__(
        self,
        pgn_path,
        search_depth,
        output_path,
        stockfish_path="stockfish",
        num_positions=None,
    ):
        self.pgn_path = pgn_path
        self.search_depth = search_depth
        self.output_path = output_path
        self.stockfish_path = stockfish_path
        self.num_positions = num_positions
        self.position_count = 0
        self.stockfish_process = None
        self._init_stockfish()

    def _init_stockfish(self):
        self.stockfish_process = subprocess.Popen(
            [self.stockfish_path],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,  
            text=True,
            bufsize=1,
        )
        self.stockfish_process.stdin.write("uci\n")
        self.stockfish_process.stdin.flush()
        while True:
            line = self.stockfish_process.stdout.readline()
            if line.startswith("uciok"):
                break
        self.stockfish_process.stdin.write("setoption name Threads value 1\n")
        self.stockfish_process.stdin.write("isready\n")
        self.stockfish_process.stdin.flush()
        while True:
            if self.stockfish_process.stdout.readline().strip() == "readyok":
                break
        print("Stockfish engine initialized")

    def read_games_(self):
        dctx = zstd.ZstdDecompressor()
        with open(self.pgn_path, "rb") as f_in:
            total_bytes = f_in.seek(0, 2) or None
            f_in.seek(0)
            with dctx.stream_reader(f_in) as reader:
                text_stream = io.TextIOWrapper(reader, encoding="utf-8", errors="ignore")
                with tqdm(desc="Games", unit=" games", dynamic_ncols=True) as game_bar:
                    while True:
                        try:
                            game = chess.pgn.read_game(text_stream)
                            if game is None:
                                break
                            game_bar.update(1)
                            yield game
                        except Exception as e:
                            print(f"[WARN] Skipping malformed game: {e}")
                            continue

    def read_positions_(self, game):
        board = chess.Board()
        try:
            for move in game.mainline_moves():
                board.push(move)
                yield board.fen()     
        except Exception as e:
            print(f"[WARN] Error mid-game, skipping rest: {e}")

    def evaluate_with_stockfish(self, fen: str) -> dict:
        try:
            stdin = self.stockfish_process.stdin
            stdout = self.stockfish_process.stdout

            stdin.write(f"position fen {fen}\ngo depth {self.search_depth}\n")
            stdin.flush()

            eval_score = 0.0
            bestmove = None

            while True:
                line = stdout.readline()
                if line.startswith("info") and " score cp " in line:
                    parts = line.split()
                    try:
                        cp_idx = parts.index("cp")
                        eval_score = float(parts[cp_idx + 1]) / 100.0
                    except (ValueError, IndexError):
                        pass
                elif line.startswith("bestmove"):
                    parts = line.split()
                    raw = parts[1] if len(parts) > 1 else None
                    bestmove = None if raw in (None, "(none)", "0000") else raw
                    break

            return {"fen": fen, "eval": eval_score, "bestmove": bestmove}

        except Exception as e:
            print(f"[ERROR] Stockfish failed for FEN {fen}: {e}")
            return {"fen": fen, "eval": 0.0, "bestmove": None}

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        if worker_info:
            worker_id = worker_info.id
            num_workers = worker_info.num_workers
        else:
            worker_id = 0
            num_workers = 1

        game_id = 0
        for game in self.read_games_():
            if game_id % num_workers != worker_id:
                game_id += 1
                continue

            for fen in self.read_positions_(game):
                if self.num_positions and self.position_count >= self.num_positions:
                    return
                result = self.evaluate_with_stockfish(fen)
                self.position_count += 1
                yield result

            del game
            gc.collect()
            game_id += 1

    def process(self, batch_size: int = 100):
        Path(self.output_path).parent.mkdir(parents=True, exist_ok=True)

        with open(self.output_path, "a", buffering=1 << 20) as output_file:
            batch: list[str] = []
            pos_bar = tqdm(
                desc="Positions",
                unit=" pos",
                total=self.num_positions,
                dynamic_ncols=True,
                smoothing=0.05,
            )
            try:
                for position_data in self:
                    batch.append(json.dumps(position_data, separators=(",", ":")))
                    pos_bar.update(1)
                    pos_bar.set_postfix(
                        eval=f"{position_data['eval']:+.2f}",
                        move=position_data["bestmove"] or "—",
                    )

                    if len(batch) >= batch_size:
                        output_file.write("\n".join(batch) + "\n")
                        output_file.flush()
                        batch.clear()
                        gc.collect()

                if batch:
                    output_file.write("\n".join(batch) + "\n")
                    output_file.flush()

            finally:
                pos_bar.close()
                self._cleanup_stockfish()

        print(f"Completed. Total positions: {self.position_count}")

    def _cleanup_stockfish(self):
        if self.stockfish_process:
            try:
                self.stockfish_process.stdin.write("quit\n")
                self.stockfish_process.stdin.flush()
            except Exception:
                pass
            self.stockfish_process.terminate()
            try:
                self.stockfish_process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                self.stockfish_process.kill()
            self.stockfish_process = None

    def __del__(self):
        self._cleanup_stockfish()

In [ ]:
processor = PGNProcessor(
    pgn_path=PGN_PATH,
    search_depth=10,
    output_path=JSONL_PATH,
    num_positions=5_000_000
)

print("PGNProcessor initialized successfully")
print(f"  PGN file: {processor.pgn_path}")
print(f"  Output: {processor.output_path}")
print(f"  Depth: {processor.search_depth}")
print(f"  Max positions: {processor.num_positions}")

In [ ]:
processor.process(batch_size=100)

## Data Pipeline
After preparing and processing data, it is necessary to read it into PyTorch data pipeline

In [ ]:
def move_to_index(move_uci: str) -> int:
    if not move_uci or len(move_uci) < 4:
        return -1
    move_uci = move_uci[:4]
    try:
        src = (ord(move_uci[0]) - ord('a')) + (8 - int(move_uci[1])) * 8
        dst = (ord(move_uci[2]) - ord('a')) + (8 - int(move_uci[3])) * 8
        return src * 64 + dst
    except (ValueError, IndexError):
        return -1

In [ ]:
def index_to_move(index: int) -> str:
    src, dst = divmod(index, 64)
    return (
        f"{chr(ord('a') + src % 8)}{8 - src // 8}"
        f"{chr(ord('a') + dst % 8)}{8 - dst // 8}"
    )

In [ ]:
test_move = "e2e4"
idx = move_to_index(test_move)
back = index_to_move(idx)
print(f"{test_move} -> {idx} -> {back}")

In [ ]:
class ChessDataset(torch.utils.data.IterableDataset):
    def __init__(
        self,
        jsonl_path,
        split = "train", # 'train/'val'
        train_ratio = 0.8,
        shuffle = False,
        sample_rate = 1.0 
    ):
        self.jsonl_path = jsonl_path
        self.split = split
        self.train_ratio = train_ratio
        self.shuffle = shuffle
        self.sample_rate = sample_rate

        self.total_lines = self._count_lines()

        split_point = int(self.total_lines * train_ratio)
        if split == "train":
            self.start_line = 0
            self.end_line = split_point
        else:
            self.start_line = split_point
            self.end_line = self.total_lines
    
    def _count_lines(self):
        with open(self.jsonl_path, 'r') as f:
            return sum(1 for _ in f)
    
    def __len__(self):
        return self.end_line - self.start_line

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        if worker_info:
            worker_id = worker_info.id
            num_workers = worker_info.num_workers
        else:
            worker_id = 0
            num_workers = 1
        
        line_num = 0
        with open(self.jsonl_path, 'r') as f:
            for line in f:
                if not (self.start_line <= line_num < self.end_line):
                    line_num += 1
                    continue
            
                if line_num % num_workers != worker_id:
                    line_num += 1
                    continue

                if np.random.random() > self.sample_rate:
                    line_num += 1
                    continue
                
                try:
                    data = json.loads(line.strip())
                    
                    if data['bestmove']:
                        data['move_idx'] = move_to_index(data['bestmove'])
                    else:
                        data['move_idx'] = -1
                    
                    yield data
                except json.JSONDecodeError:
                    print(f"[WARN] skipping malformed JSON at line {line_num}")
                except (ValueError, IndexError) as e:
                    print(f"[WARN] failed to convert move at line {line_num}: {e}")
                
                line_num += 1

In [ ]:
print("Creating train/val datasets...")

train_dataset = ChessDataset(
    jsonl_path=JSONL_PATH,
    split='train',
    train_ratio=0.8,
    sample_rate=1.0
)

val_dataset = ChessDataset(
    jsonl_path=JSONL_PATH,
    split='val',
    train_ratio=0.8,
    sample_rate=1.0
)

print(f"Train dataset: {len(train_dataset)} positions")
print(f"Val dataset: {len(val_dataset)} positions")

## Defining and creating a graph structure

In [ ]:
class ChessGraphEncoder:
    PIECE_TO_IDX = {
        chess.PAWN: 0, chess.KNIGHT: 1, chess.BISHOP: 2,
        chess.ROOK: 3, chess.QUEEN: 4,  chess.KING: 5,
    }
    PIECE_VALUES = {
        chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
        chess.ROOK: 5, chess.QUEEN: 9,  chess.KING: 0,
    }
    _ADJACENCY_CACHE= None


    def _node_features(self, board: chess.Board) -> torch.Tensor:
        features = []
        attacked_by_opp = {sq for sq in range(64) if board.is_attacked_by(not board.turn, sq)}

        for sq in range(64):
            piece = board.piece_at(sq)
            one_hot = torch.zeros(12)
            if piece is not None:
                one_hot[self.PIECE_TO_IDX[piece.piece_type] + (0 if piece.color == chess.WHITE else 6)] = 1.0

            activity = 0.0
            if piece is not None and piece.color == board.turn:
                n_moves = sum(1 for m in board.legal_moves if m.from_square == sq)
                activity = min(n_moves / 8.0, 1.0)

            features.append(torch.cat([
                one_hot,
                torch.tensor([
                    (sq // 8) / 7.0,
                    (sq %  8) / 7.0,
                    float(sq in attacked_by_opp),
                    float(board.is_pinned(board.turn, sq)),
                    activity,
                ]),
            ]))
        return torch.stack(features)  # (64, 17)

    def _attack_edges(self, board: chess.Board):
        checking_moves = {m.from_square for m in board.legal_moves if board.gives_check(m)}

        edge_list, edge_attrs = [], []
        for move in board.legal_moves:
            fs, ts = move.from_square, move.to_square
            target = board.piece_at(ts)
            if not (board.is_capture(move) or fs in checking_moves or target is not None):
                continue

            dist = max(abs(fs // 8 - ts // 8), abs(fs % 8 - ts % 8))
            cap_val = self.PIECE_VALUES[target.piece_type] / 9.0 if board.is_capture(move) and target else 0.0
            edge_list.append([fs, ts])
            edge_attrs.append([dist / 7.0, cap_val, float(board.gives_check(move))])

        if not edge_list:
            return torch.zeros((2, 0), dtype=torch.long), torch.zeros((0, 3))
        return (
            torch.tensor(edge_list, dtype=torch.long).t().contiguous(),
            torch.tensor(edge_attrs, dtype=torch.float32),
        )

    def _defence_edges(self, board: chess.Board):
        edge_list, edge_attrs = [], []
        for fs in range(64):
            piece = board.piece_at(fs)
            if piece is None:
                continue
            for ts in range(64):
                target = board.piece_at(ts)
                if target is None or target.color != piece.color or fs == ts:
                    continue
                if board.is_attacked_by(piece.color, ts):
                    dist = max(abs(fs // 8 - ts // 8), abs(fs % 8 - ts % 8))
                    edge_list.append([fs, ts])
                    edge_attrs.append([
                        dist / 7.0,
                        self.PIECE_VALUES[piece.piece_type]  / 9.0,
                        self.PIECE_VALUES[target.piece_type] / 9.0,
                    ])

        if not edge_list:
            return torch.zeros((2, 0), dtype=torch.long), torch.zeros((0, 3))
        return (
            torch.tensor(edge_list, dtype=torch.long).t().contiguous(),
            torch.tensor(edge_attrs, dtype=torch.float32),
        )

    @classmethod
    def _get_adjacency_edges(cls) -> tuple[torch.Tensor, torch.Tensor]:
        if cls._ADJACENCY_CACHE is not None:
            return cls._ADJACENCY_CACHE

        directions = [
            (-1, 0), (1, 0), (0, -1), (0, 1),
            (-1, -1), (-1, 1), (1, -1), (1, 1),
        ]
        edge_list, edge_attrs = [], []
        for from_sq in range(64):
            fr, ff = divmod(from_sq, 8)
            for dir_idx, (dr, df) in enumerate(directions):
                r, f = fr + dr, ff + df
                while 0 <= r < 8 and 0 <= f < 8:
                    one_hot = torch.zeros(8)
                    one_hot[dir_idx] = 1.0
                    edge_list.append([from_sq, r * 8 + f])
                    edge_attrs.append(one_hot)
                    r += dr; f += df

        cls._ADJACENCY_CACHE = (
            torch.tensor(edge_list, dtype=torch.long).t().contiguous(),
            torch.stack(edge_attrs),
        )
        return cls._ADJACENCY_CACHE

    def _global_context(self, board: chess.Board) -> torch.Tensor:
        ctx = torch.tensor([
            float(board.has_kingside_castling_rights(chess.WHITE)),
            float(board.has_queenside_castling_rights(chess.WHITE)),
            float(board.has_kingside_castling_rights(chess.BLACK)),
            float(board.has_queenside_castling_rights(chess.BLACK)),
            (board.ep_square % 8) / 8.0 if board.ep_square else 0.0,
            float(board.turn),
            min(board.halfmove_clock  / 100.0, 1.0),
            min(board.fullmove_number / 100.0, 1.0),
        ], dtype=torch.float32)
        return F.pad(ctx, (0, 17 - ctx.shape[0]))  # pad to 17

    def encode_fen(self, fen: str) -> HeteroData:
        board = chess.Board(fen)
        graph = HeteroData()

        node_feats  = self._node_features(board)
        global_node = self._global_context(board)
        graph['square'].x = torch.cat([node_feats, global_node.unsqueeze(0)])  # (65, 17)

        atk_ei, atk_ea = self._attack_edges(board)
        def_ei, def_ea = self._defence_edges(board)
        adj_ei, adj_ea = self._get_adjacency_edges()

        if atk_ei.numel():
            graph['square', 'attack',  'square'].edge_index = atk_ei
            graph['square', 'attack',  'square'].edge_attr  = atk_ea
        if def_ei.numel():
            graph['square', 'defence', 'square'].edge_index = def_ei
            graph['square', 'defence', 'square'].edge_attr  = def_ea

        graph['square', 'adjacent', 'square'].edge_index = adj_ei
        graph['square', 'adjacent', 'square'].edge_attr  = adj_ea

        return graph

In [ ]:
_ENCODER = ChessGraphEncoder() 

def collate_graphs(batch: list[dict]):
    graphs, evals, move_idxs = [], [], []
    for item in batch:
        graphs.append(_ENCODER.encode_fen(item['fen']))
        evals.append(item['eval'])
        move_idxs.append(item.get('move_idx', -1))

    return {
        'graphs':     Batch.from_data_list(graphs),
        'evals':      torch.tensor(evals,     dtype=torch.float32),
        'move_idxs':  torch.tensor(move_idxs, dtype=torch.long),
    }

In [ ]:
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0,  
    collate_fn=collate_graphs,
    drop_last=False
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0,
    collate_fn=collate_graphs,
    drop_last=False
)

## Model definition

In [ ]:
class GATeauLayer(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, num_heads: int):
        super().__init__()
        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)

        edge_feat_dims = {"attack": 3, "defence": 3, "adjacent": 8}
        self.edge_proj = nn.ModuleDict({
            name: nn.Linear(dim, num_heads)
            for name, dim in edge_feat_dims.items()
        })

        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(0.1)

    def _edge_attention(
        self,
        Q: torch.Tensor,          # (N, H, D)
        K: torch.Tensor,          # (N, H, D)
        V: torch.Tensor,          # (N, H, D)
        edge_attr: torch.Tensor,  # (E, feat_dim)
        src: torch.Tensor,        # (E,)
        dst: torch.Tensor,        # (E,)
        edge_type: str,
        num_nodes: int,
    ) -> torch.Tensor:            # (N, H, D)
        scores = (Q[dst] * K[src]).sum(-1) * self.scale       # (E, H)
        scores = scores + self.edge_proj[edge_type](edge_attr) # (E, H)

        max_score = torch.full((num_nodes, self.num_heads), float("-inf"), device=Q.device)
        max_score.scatter_reduce_(0, dst.unsqueeze(-1).expand_as(scores), scores, reduce="amax", include_self=True)
        scores = scores - max_score[dst]  # (E, H)

        exp_scores = scores.exp()         # (E, H)
        sum_exp = torch.zeros(num_nodes, self.num_heads, device=Q.device)
        sum_exp.scatter_add_(0, dst.unsqueeze(-1).expand_as(exp_scores), exp_scores)

        attn = exp_scores / (sum_exp[dst] + 1e-8)  # (E, H)

        weighted = V[src] * attn.unsqueeze(-1)      # (E, H, D)
        out = torch.zeros(num_nodes, self.num_heads, self.head_dim, device=Q.device)
        idx = dst[:, None, None].expand_as(weighted)
        out.scatter_add_(0, idx, weighted)
        return out  # (N, H, D)

    def forward(
        self,
        node_features: torch.Tensor,  # (N, in_dim)
        edge_dict: dict,
    ) -> torch.Tensor:                # (N, hidden_dim)
        num_nodes = node_features.shape[0]

        x = self.input_proj(node_features)
        Q = self.q_proj(x).reshape(num_nodes, self.num_heads, self.head_dim)
        K = self.k_proj(x).reshape(num_nodes, self.num_heads, self.head_dim)
        V = self.v_proj(x).reshape(num_nodes, self.num_heads, self.head_dim)

        attn_output = torch.zeros(num_nodes, self.num_heads, self.head_dim, device=x.device)

        for edge_type in ("attack", "defence", "adjacent"):
            entry = edge_dict.get(edge_type)
            if entry is None:
                continue
            edge_attr, src, dst = entry
            if src.shape[0] == 0:
                continue

            attn_output += self._edge_attention(Q, K, V, edge_attr, src, dst, edge_type, num_nodes)

        out = self.out_proj(attn_output.reshape(num_nodes, self.hidden_dim))
        return self.dropout(self.norm(out + x))

In [ ]:
class DSMoELayer(nn.Module):
    def __init__(self, in_dim, expert_dim, num_experts, top_k = 1):
        super().__init__()
        assert 1 <= top_k <= num_experts, "top_k must be between 1 and num_experts"

        self.in_dim = in_dim
        self.num_experts = num_experts
        self.top_k = top_k

        self.router = nn.Linear(in_dim, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(in_dim, expert_dim),
                nn.ReLU(),
                nn.Linear(expert_dim, in_dim)
            )
            for _ in range(num_experts)
        ])
        self.dropout = nn.Dropout(0.1)

    def _dense_forward(self, x, weights):
        expert_outs = torch.stack([expert(x) for expert in self.experts], dim=0)
        return (weights.T.unsqueeze(-1) * expert_outs).sum(0)
    
    def _sparse_forward(self, x, weights):
        N = x.shape[0]
        topk_weights, topk_idx = weights.topk(self.top_k, dim=1)
        topk_weights = topk_weights / topk_weights.sum(dim=1, keepdim=True) #normalize

        output = torch.zeros_like(x)
        for k in range(self.top_k):
            expert_idx = topk_idx[:,k]
            for e_id in expert_idx.unique():
                node_mask = expert_idx == e_id
                output[node_mask] += (
                    topk_weights[node_mask, k].unsqueeze(-1) * self.experts[e_id](x[node_mask])
                )
            
        return output
    
    def forward(self, x):
        weights = F.softmax(self.router(x), dim=1)
        out = self._dense_forward(x, weights) if self.training else self._sparse_forward(x, weights)
        return self.dropout(out)

In [ ]:
class SAGEModel(nn.Module):
    EDGE_TYPES = {
        "attack":   ("square", "attack",   "square"),
        "defence":  ("square", "defence",  "square"),
        "adjacent": ("square", "adjacent", "square"),
    }

    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_layers,
        num_heads,
        num_experts,
        expert_dim,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.input_proj = nn.Linear(input_dim, hidden_dim)

        self.layers = nn.ModuleList([
            nn.ModuleDict({
                "gateau": GATeauLayer(hidden_dim, hidden_dim, num_heads),
                "moe":    DSMoELayer(hidden_dim, expert_dim, num_experts),
            })
            for _ in range(num_layers)
        ])

        self.value_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )

        self.policy_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 4096),
        )

    @staticmethod
    def _extract_edge_dict(batch):
        edge_dict = {}
        for key, triplet in SAGEModel.EDGE_TYPES.items():
            if triplet in batch.edge_types:
                ei   = batch[triplet].edge_index
                attr = batch[triplet].edge_attr
                edge_dict[key] = (attr, ei[0], ei[1])
            else:
                edge_dict[key] = None
        return edge_dict

    @staticmethod
    def _global_node_indices(batch):
        batch_size  = batch.num_graphs
        nodes_per_graph = batch["square"].num_nodes // batch_size
        return torch.arange(batch_size, device=batch["square"].x.device) * nodes_per_graph + (nodes_per_graph - 1)

    def forward(self, batch_graphs) -> tuple[torch.Tensor, torch.Tensor]:
        if isinstance(batch_graphs, list):
            batch = Batch.from_data_list(batch_graphs)
        else:
            batch = batch_graphs

        x = self.input_proj(batch["square"].x)   # (N_total, hidden_dim)
        edge_dict = self._extract_edge_dict(batch)

        for layer in self.layers:
            x = layer["gateau"](x, edge_dict)
            x = layer["moe"](x)

        global_idx = self._global_node_indices(batch)    # (batch_size,)
        global_repr = x[global_idx]                      # (batch_size, hidden_dim)

        values   = self.value_head(global_repr).squeeze(-1)  # (batch_size,)
        policies = self.policy_head(global_repr)              # (batch_size, 4096)

        return values, policies

In [ ]:
model = SAGEModel(
    input_dim=17,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    num_experts=NUM_EXPERTS,
    expert_dim=EXPERT_DIM
)

model = model.to(DEVICE)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# Training the model

In [ ]:
class Trainer:
    def __init__(
        self,
        model,
        device,
        lr,
        value_weight,
        policy_weight,
        num_epochs,
        checkpoint_dir
    ):
        self.model = model.to(device)
        self.device = device
        self.value_weight = value_weight
        self.policy_weight = policy_weight
        self.checkpoint_dir = checkpoint_dir
        checkpoint_dir.mkdir(parents=True, exist_ok=True)

        self.optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=num_epochs)

        self.value_loss_fn = nn.MSELoss()
        self.policy_loss_fn = nn.CrossEntropyLoss(reduction='none')

        self.history: dict[str, list] = {
            'train_loss': [], 'train_value_mae': [], 'train_policy_acc': [],
            'val_loss':   [], 'val_value_mae':   [], 'val_policy_acc':   [],
        }
    
    def _compute_losses(
        self,
        values_pred,
        values_true,
        policies_pred,
        move_idxs
    ):
        loss_value = self.value_loss_fn(values_pred, values_true)

        valid_mask = move_idxs >= 0
        if valid_mask.any():
            policy_loss_all = self.policy_loss_fn(policies_pred, move_idxs.clamp(min=0))
            loss_policy = policy_loss_all[valid_mask].mean()
        else:
            loss_policy = torch.tensor(0.0, device=self.device)

        loss_total = self.value_weight * loss_value + self.policy_weight * loss_policy
        return loss_total, loss_value, loss_policy

    def _compute_metrics(
        self,
        values_pred,
        values_true,
        policies_pred,
        move_idxs
    ):
        value_mae = (values_pred - values_true).abs().mean().item() * 100 # centipawns
        valid_mask = move_idxs >= 0
        if valid_mask.any():
            policy_acc = (
                policies_pred[valid_mask].argmax(dim=1) == move_idxs[valid_mask]
            ).float().mean().item()
        else:
            policy_acc = 0.0

        return value_mae, policy_acc

    def _run_epoch(self, loader, *, training: bool) -> tuple[float, float, float]:
        self.model.train(training)
        total_loss = total_mae = total_acc = 0.0
        desc = "Train" if training else "Val"

        with torch.set_grad_enabled(training):
            pbar = tqdm(loader, desc=desc, leave=True, dynamic_ncols=True)
            for i, batch_data in enumerate(pbar, 1):
                graphs    = batch_data['graphs'].to(self.device)
                evals     = batch_data['evals'].to(self.device)
                move_idxs = batch_data['move_idxs'].to(self.device)

                values_pred, policies_pred = self.model(graphs)
                loss, _, _ = self._compute_losses(values_pred, evals, policies_pred, move_idxs)

                if training:
                    self.optimizer.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    self.optimizer.step()

                mae, acc = self._compute_metrics(
                    values_pred.detach(), evals, policies_pred.detach(), move_idxs
                )
                total_loss += loss.item()
                total_mae  += mae
                total_acc  += acc

                pbar.set_postfix({
                    'loss': f'{total_loss / i:.4f}',
                    'mae':  f'{total_mae  / i:.1f}cp',
                    'acc':  f'{total_acc  / i:.3f}',
                    'lr':   f'{self.scheduler.get_last_lr()[0]:.2e}' if training else '',
                })

        n = len(loader)
        return total_loss / n, total_mae / n, total_acc / n
    
    def _save_checkpoint(self, epoch: int, val_loss: float, tag: str = "best"):
        path = self.checkpoint_dir / f"{tag}_epoch{epoch}_val{val_loss:.4f}.pt"
        torch.save({
            'epoch':           epoch,
            'model_state':     self.model.state_dict(),
            'optimizer_state': self.optimizer.state_dict(),
            'scheduler_state': self.scheduler.state_dict(),
            'val_loss':        val_loss,
            'history':         self.history,
        }, path)
        last = self.checkpoint_dir / f"{tag}.pt"
        if last.exists() or last.is_symlink():
            last.unlink()
        last.symlink_to(path.name)
        return path

    def train(self, train_loader, val_loader, num_epochs: int):
        best_val_loss = float('inf')

        for epoch in range(1, num_epochs + 1):
            print(f"\n{'='*60}\nEpoch {epoch}/{num_epochs}\n{'='*60}")

            train_loss, train_mae, train_acc = self._run_epoch(train_loader, training=True)
            self.scheduler.step()

            val_loss, val_mae, val_acc = self._run_epoch(val_loader, training=False)

            for key, val in zip(
                ['train_loss','train_value_mae','train_policy_acc',
                 'val_loss',  'val_value_mae',  'val_policy_acc'],
                [train_loss, train_mae, train_acc, val_loss, val_mae, val_acc],
            ):
                self.history[key].append(val)

            print(f"Train | Loss: {train_loss:.4f} | MAE: {train_mae:.1f}cp | Acc: {train_acc:.3f}")
            print(f"Val   | Loss: {val_loss:.4f}   | MAE: {val_mae:.1f}cp   | Acc: {val_acc:.3f}")
            print(f"LR: {self.scheduler.get_last_lr()[0]:.2e}")

            self._save_checkpoint(epoch, val_loss, tag="last")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                path = self._save_checkpoint(epoch, val_loss, tag="best")
                print(f"Best checkpoint: {path.name}")

        print(f"\n{'='*60}\nDone. Best val loss: {best_val_loss:.4f}\n{'='*60}")

In [ ]:
trainer = Trainer(
    model=model,
    device=DEVICE,
    lr=LEARNING_RATE,
    value_weight=VALUE_WEIGHT,
    policy_weight=POLICY_WEIGHT,
    num_epochs=NUM_EPOCHS,
    checkpoint_dir=CHECKPOINT_DIR,
)
trainer.train(train_loader, val_loader, NUM_EPOCHS)